# Automatic Deep Research 

Welcome to this new practice lab! By now you should have a clearer view of the elements that compose a multi-agent system. In this lab you will get to put it into action by creating your first crew.

**What you'll learn:**
- How to define agents with specific roles and expertise
- How to provide agents with tools to perform their tasks
- How to create your own tasks that agents will execute
- How to assemble agents and tasks into a Crew, all using CrewAI

## Background

As a research consultant, you're constantly tasked with producing comprehensive reports on diverse topics for demanding clients. You need to build an automatic deep research solution that can rapidly gather, verify, and synthesize information from across the internet, delivering reliable, fact-checked reports that meet tight deadlines and exacting standards regardless of the subject matter. 

## General instructions
In this lab you will be presented with a structure of the code, but you will need to complete some of it. 

To successfully run this lab, replace all instances of the placeholder `None` with your own code. Sections where you need to write code will be delimited between `### START CODE HERE ###` and `### END CODE HERE ###`.

If you are stuck, or simply want to copy a solution into your notebook so that you can execute it, you can find all solution code inside the [Solution](Solution) folder.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of contents

- [1. Understanding the problem](#1)
- [2. Set up your notebook](#2)
- [3. Define the Agents](#3)
  - [3.1. Create tool instances](#3-1)
  - [3.2. Define the Research Planner agent](#3-2)
  - [3.3. Define the remaining agents](#3-3)
- [4. Create the Tasks](#4)
  - [4.1. Define the Create research plan task](#4-1)
  - [4.2. Define the remaining tasks](#4-2)
- [5. Define the Crew and get the results](#5)

<a id="1"></a>

## 1. Understanding the problem
In this lab, you will focus on building a custom deep research crew. This Crew will be in charge of creating a research plan based on the user's input, and executing it, while reviewing and checking the facts. Finally, with the gathered information a report needs to be generated.

Take some time to decompose the problem into different tasks. Who would be the appropriate "person" to solve each task? 

Once you've done your thinking, click below to find an agent/task diagram for this lab.    


<details>    
<summary>
    <font size="3" color="#237b946b"><b>Diagram</b></font>
</summary>

<img src="../images/lab2-agents-tasks-diagram.PNG">

<a id="2"></a>

## 2. Set up your notebook

Before you start coding, run the next two cells to import all necessary modules and configure the environment variables. 

In [11]:
from crewai import Agent, Task, Crew
import os
from datetime import date
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key
os.environ["OPENAI_API_KEY"] = get_openai_api_key()

# Today's date is injected into the tasks so the agents always know
# what "recent" / "latest" / "this week" actually means when they search.
TODAY = date.today().isoformat()
print("Model:", os.environ["MODEL"])
print("Today's date (used for recency grounding):", TODAY)

Model: gpt-4o-mini
Today's date (used for recency grounding): 2026-08-15


<a id="3"></a>

## 3. Define the Agents

Based on the diagram, you should have four agents:
- **Research Planner**: its goal is to analyze queries and break them down into smaller, specific research topics.
- **Internet Researcher**: its job is to perform research tasks.
- **Fact checker**: its goal is to review information for fact accuracy to avoid misinformation. 
- **Report Writer**: is in charge of writing reports, based on gathered information.

<a id="3-1"></a>

### 3.1. Create tool instances
As you can see in the diagram, you will be providing the **Internet Researcher Agent** with tools, so that it can better do their job. In particular, you will give this agent access to search the internet and scrape information from the retrieved webpages. 

There are different tools inside CrewAI you can use to search the web, in this lab you will use the [**EXA Search Web Loader**](https://docs.crewai.com/en/tools/search-research/exasearchtool#exa-search-web-loader) tool, which is designed to perform a semantic search for a specified query from a text’s content across the internet. It utilizes the [exa.ai](https://exa.ai/) API to fetch and display the most relevant search results based on the query provided by the user. exa.ai enhances semantic search by capturing richer contextual relationships between concepts, allowing for more precise information retrieval than conventional embedding approaches.

For webscraping, you will use the [**Scrape Website**](https://docs.crewai.com/en/tools/web-scraping/scrapewebsitetool) tool, which is designed to extract and read the content of a specified website.

In the next cell you will define instances of these tools, so you can later assign them to the agents.

In [2]:
# import the tools
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
from utils import get_exa_api_key

# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

# Create the EXASearchTool instance (web + news semantic search)
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))
# Create the ScrapeWebsiteTool instance (full-page content extraction)
scrape_website_tool = ScrapeWebsiteTool()

<a id="3-2"></a>

### 3.2. Define the Research Planner agent

In the cell below, you will see how you can create the first agent. This time, all the parameters are set up for you. Here is a quick recap of what each of the parameters represent:

- `Role`: If this was a person doing the job, what title would they have?
- `Goal`: What is the goal this agent in particular is trying to accomplish? Make sure to write concrete goal
- `Background`: it should be something the highlights the skills of the agent relevant to its role. Make sure to use keywords that will actually help your agent get better results.

In the labs, we have added two parameters not shown in the demo videos: `max_rpm`, and `max_iter`. `max_rpm` sets the maximum requests per minute to avoid rate limits, while `max_iter` limits the maximum iterations before the agent must provide its best answer. Setting these two parameters helps make the agents run a little faster, so the lab doesn't take as long to complete. 

In [3]:
query_analyzer = Agent(
    role="Fact-Check Query Analyzer",
    goal=(
        "Break down the user's query or claim into specific, checkable sub-claims, "
        "identify the exact entities, events, and dates involved, and determine the "
        "correct time window to search (a specific date/period if the user gave one, "
        "otherwise the most recent available information as of {current_date})."
    ),
    backstory=(
        "You are a fact-checking desk editor, similar to those at outlets like TN Fact Check / "
        "TN IID, who specializes in turning vague or loaded user queries into precise, verifiable "
        "research questions. You are extremely careful about time: you always distinguish between "
        "'what happened historically' and 'what is happening now', and you flag explicitly when a "
        "query needs today's or this week's news rather than background information. You never let "
        "an ambiguous claim go unresolved -- you always specify exactly what needs to be verified "
        "and by when the underlying information should be dated."
    ),
    verbose=True,
    max_rpm=150,
    max_iter=15
)

<a id="3-3"></a>

### 3.3. Define the remaining agents

Now you can define the three remaining agents. The `role` and `goal` parameters are already filled in for you; use your own creativity to fill in the `backstory`.  

Do not forget to assign the tools to the **Internet Researcher** and **Fact Checker** agents. You can do this by setting the `tools` argument.

In [4]:
news_researcher = Agent(
    role="Real-Time News Researcher",
    goal=(
        "Retrieve the most recent, relevant information available for each research "
        "topic -- prioritizing breaking news, official statements, recent actions, and "
        "reports published within the requested (or most recent possible) time window -- "
        "and record the exact publish date and source URL for everything found."
    ),
    backstory=(
        "You are an investigative wire-service researcher who specializes in real-time "
        "news retrieval. You know that yesterday's article is more useful than last year's, "
        "so you always search with recency-biased terms ('latest', 'today', 'this week', "
        "specific months/years) and sort mentally for the newest credible coverage first. "
        "You cross-reference multiple outlets (news sites, official government/organization "
        "pages, press releases) rather than relying on a single source, and you scrape pages "
        "when a snippet is not enough to confirm a date, figure, or direct quote."
    ),
    tools=[exa_search_tool, scrape_website_tool],
    verbose=True,
    max_rpm=150,
    max_iter=15
)

fact_verifier = Agent(
    role="Fact Verification Specialist",
    goal=(
        "Verify every claim and figure gathered by the researcher against at least one "
        "independent, credible, and recent source; flag information that is outdated, "
        "unsupported, contradictory, or based on a single unverified source; and rate "
        "the recency and reliability of each source used."
    ),
    backstory=(
        "You are a meticulous fact-checking specialist in the mold of TN Fact Check / TN IID "
        "verification desks. You never accept a claim at face value -- you actively re-search "
        "and cross-check it against independent sources, paying special attention to publish "
        "dates so that outdated information is never presented as current. You clearly label "
        "each claim as Verified, Partially Verified, Unverified, Outdated, or False, and you "
        "explain exactly why."
    ),
    tools=[exa_search_tool, scrape_website_tool],
    verbose=True,
    max_rpm=150,
    max_iter=15
)

factcheck_report_writer = Agent(
    role="Fact-Check Report Writer",
    goal=(
        "Write a clear, well-structured fact-check report that gives the user a direct "
        "verdict on their query, backed by the verified and up-to-date evidence, with "
        "full source citations including publish dates."
    ),
    backstory=(
        "You are a professional fact-check report writer who transforms verification "
        "findings into a concise, publication-ready verdict -- similar to how outlets "
        "like TN Fact Check / TN IID present their conclusions. You lead with a clear "
        "rating (True / False / Misleading / Unverified / Needs More Context), summarize "
        "the evidence in plain language, explicitly note how recent the information is, "
        "and list every source with a link and publish date."
    ),
    verbose=True,
    max_rpm=150,
    max_iter=15
)

<a id="4"></a>

## 4. Create the Tasks

Now that you have set up the agents, it is time to define the tasks. If you go back to the diagram, you will see you need four tasks:

- **Create research plan**: Based on the user's query, break it down into specific topics and key questions, and create a focused research plan.
    - Output: A research plan with main research topics to investigate, key questions for each topic, and success criteria for the research.

- **Gather research data**: Using the research plan, collect information on all identified topics. Cite all sources used.
    - Output: Comprehensive research data including: information for each research topic, and citations used along with source credibility notes.

- **Verify information quality**: Review all collected research. Identify any conflicting information, potential misinformation, or gaps that need addressing.
    - Output: A report with the all the collected data, and its review. It should include consistency check results and source reliability ratings

- **Write final report**: Create a comprehensive report that answers the original query using all verified research data. Structure it with clear sections, include citations, and provide actionable insights.
    - Output: The final research report. In addition to the full answer, it should have an executive summary, and complete source citations.


For each `Task` you need to define the following parameters:
- `description`: A thorough description of the task. You can even break it down into different items.
- `expected_output`: what should the output return. Be specific, specially if you want any structure in your result, like a dictionary with specific keys.
- `agent`: who is performing the task? You need to match the task to one of the agents you already defined

In the description you will need to pass the inputs to the tasks. In this lab, you will only have as input the user's query, which will be saved as `user_query`:


<a id="4-1"></a>

### 4.1. Define the Create research plan task

In the cell below, you will see how you can create the first task. This time, all the parameters are set up for you. Notice how the context variables are passed the the description between curly brackets. 

In [5]:
analyze_query_task = Task(
    description=(
        "Analyze the user's query and break it down into specific, checkable sub-claims "
        "and key questions. Identify all entities, events, organizations, and dates involved. "
        "Determine the correct time window for research: if the user specified a date or period, "
        "use exactly that; otherwise, target the most recent information available as of "
        "{current_date}. Produce a focused research plan with concrete, recency-biased search "
        "queries (e.g. including terms like 'latest', 'today', 'this week', or the relevant year/month) "
        "for the researcher to use.\n\n"
        "The user's query is: {user_query}\n"
        "The user-specified time period (if any) is: {time_period}\n"
        "Today's date is: {current_date}"
    ),
    expected_output=(
        "A research plan listing: (1) the specific sub-claims/questions to verify, "
        "(2) the exact entities/events/dates involved, (3) the determined time window "
        "to search within, and (4) a set of concrete, recency-biased search queries "
        "for each sub-claim."
    ),
    agent=query_analyzer,
)

<a id="4-2"></a>

### 4.2. Define the remaining tasks

Now define the three remaining tasks. The `description` is already filled in for you, you will need to define the `expected_output` and `agent` for each of the Tasks.

In [6]:
# define the retrieve recent information task
retrieve_recent_info_task = Task(
    description=(
        "Using the research plan, search the web and news sources for the most recent, "
        "relevant information on every identified sub-claim and topic. Prioritize sources "
        "published within the determined time window -- use recency-biased search terms "
        "and, when the user gave no specific period, actively seek out the latest available "
        "news, actions, or developments rather than older background material. For every "
        "piece of information gathered, record the source URL and its publish date, and "
        "scrape the page when needed to confirm exact dates, figures, or quotes."
    ),
    expected_output=(
        "A detailed collection of research findings covering every sub-claim, each entry "
        "including the finding itself, the source URL, and the source's publish date, "
        "clearly separating the most recent findings from any older/background context."
    ),
    agent=news_researcher
)

# define the verify facts task
verify_facts_task = Task(
    description=(
        "Review all gathered research. For each claim or finding, verify it against at "
        "least one independent, credible source. Identify any conflicting information, "
        "outdated claims (information superseded by more recent developments), potential "
        "misinformation, or gaps that still need addressing. Explicitly check whether each "
        "source's publish date falls within the required time window; if a source is stale, "
        "search for a more recent update and note whether the situation has changed."
    ),
    expected_output=(
        "A fact-verification summary that, for each sub-claim, states a status "
        "(Verified / Partially Verified / Unverified / Outdated / False), the supporting "
        "evidence, any conflicting claims found, a note on source recency and reliability, "
        "and any recommended corrections or additional research needed."
    ),
    agent=fact_verifier
)

# define the write final fact-check report task
write_factcheck_report_task = Task(
    description=(
        "Create a final fact-check report that directly answers the user's original query "
        "using only the verified, up-to-date research. Lead with a clear overall verdict "
        "(True / False / Misleading / Unverified / Needs More Context). Summarize the "
        "supporting evidence in plain language, explicitly state how recent the underlying "
        "information is (mention specific dates), and list every source used with its link "
        "and publish date."
    ),
    expected_output=(
        "A comprehensive, clearly structured fact-check report containing: an overall verdict, "
        "an executive summary, a detailed evidence breakdown per sub-claim, an explicit note on "
        "the recency of the information used, and a complete list of source citations with "
        "publish dates."
    ),
    agent=factcheck_report_writer
)

<a id="5"></a>

## 5. Define the Crew and get the results

Once the agents and tasks have been defined, you are ready to create the crew. In order to so, you will need to set the following arguments:
- `agents`: list of agents in the crew
- `tasks`: list of tasks in the crew. The tasks should be listed in the order they should be executed

In the next cell, fill in the agents and tasks for the crew.

In [7]:
# create the crew with the defined agents and tasks
crew = Crew(
    agents=[query_analyzer, news_researcher, fact_verifier, factcheck_report_writer],
    tasks=[analyze_query_task, retrieve_recent_info_task, verify_facts_task, write_factcheck_report_task]
)

Before running the crew, you need to define the query, which will be used as input for the tasks.

In [8]:
# Write the query/claim you want fact-checked
user_query = "Did the Tamil Nadu government announce a hike in house/property tax?"

# Optional: pin the fact-check to a specific date/period.
# Leave as "not specified" to default to the most recent available information.
time_period = "August 2026"

Now you are only left with kickstarting the crew to get the results. Since you set `verbose=True` in the agents, you should monitor all the process.

In [ ]:
result = crew.kickoff(
    inputs={
        "user_query": user_query,
        "time_period": time_period,
        "current_date": TODAY,
    }
)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Query Analyzer                                                                               │
│                                                                                                                 │
│  Task: Analyze the user's query and break it down into specific, checkable sub-claims and key questions.        │
│  Identify all entities, events, organizations, and dates involved. Determine the correct time window for        │
│  research: if the user specified a date or period, use exactly that; otherwise, target the most recent          │
│  information available as of 2026-08-15. Produce a focused research plan with concrete, recency-biased search   │
│  queries (e.g. including terms like 'latest', 'today', 'this week', or the relevant year/month) for the         │
│  researcher to use.                                                                                             │
│                                                                                                                 │
│  The user's query is: Did the Tamil Nadu government announce a hike in house/property tax?                      │
│  The user-specified time period (if any) is: August 2026                                                        │
│  Today's date is: 2026-08-15                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Query Analyzer                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. **Specific Sub-Claims/Questions to Verify:**                                                                │
│     - Has the Tamil Nadu government officially announced a hike in house/property tax?                          │
│     - What is the percentage of the increase in the house/property tax, if announced?                           │
│     - When is this hike expected to take effect?                                                                │
│     - What are the reasons cited by the Tamil Nadu government for this tax increase?                            │
│                                                                                                                 │
│  2. **Exact Entities/Events/Dates Involved:**                                                                   │
│     - Entity: Tamil Nadu Government                                                                             │
│     - Event: Announcement of house/property tax hike                                                            │
│     - Dates: Specifically, August 2026 (as the user requested information relevant to this timeframe)           │
│                                                                                                                 │
│  3. **Determined Time Window to Search Within:**                                                                │
│     - Search for the most recent information as of August 2026, focusing on announcements and discussions from  │
│  early August 2026 up to the current date (August 15, 2026).                                                    │
│                                                                                                                 │
│  4. **Set of Concrete, Recency-Biased Search Queries:**                                                         │
│     - "Tamil Nadu government house property tax hike announcement August 2026"                                  │
│     - "Tamil Nadu property tax increase details August 2026"                                                    │
│     - "Reasons for Tamil Nadu house tax hike August 2026"                                                       │
│     - "Latest news on Tamil Nadu property tax August 2026"                                                      │
│                                                                                                                 │
│  This focused research plan will help in verifying the user's query regarding the Tamil Nadu government's       │
│  announcement about a hike in the house/property tax effectively.                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Real-Time News Researcher                                                                               │
│                                                                                                                 │
│  Task: Using the research plan, search the web and news sources for the most recent, relevant information on    │
│  every identified sub-claim and topic. Prioritize sources published within the determined time window -- use    │
│  recency-biased search terms and, when the user gave no specific period, actively seek out the latest           │
│  available news, actions, or developments rather than older background material. For every piece of             │
│  information gathered, record the source URL and its publish date, and scrape the page when needed to confirm   │
│  exact dates, figures, or quotes.                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Real-Time News Researcher                                                                               │
│                                                                                                                 │
│  Thought: I need to search for the most recent information regarding the Tamil Nadu government's announcement   │
│  about a hike in house/property tax, specifically focusing on any details, percentage increases, effective      │
│  dates, and cited reasons for the tax increase. I will utilize the recency-biased search queries defined in     │
│  the research plan.                                                                                             │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "Tamil Nadu government house property tax hike announcement August 2026",                    │
│    "start_published_date": "2026-08-01",                                                                        │
│    "end_published_date": "2026-08-15",                                                                          │
│    "include_domains": null                                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Title: Greater Chennai Corporation puts property tax revision on hold following taxpayer complaints - The      │
│  HinduBusinessLine                                                                                              │
│  URL:                                                                                                           │
│  https://www.thehindubusinessline.com/news/national/greater-chennai-corporation-puts-property-tax-revision-on-  │
│  hold-following-taxpayer-complaints/article71344198.ece                                                         │
│  ID:                                                                                                            │
│  https://www.thehindubusinessline.com/news/national/greater-chennai-corporation-puts-property-tax-revision-on-  │
│  hold-following-taxpayer-complaints/article71344198.ece                                                         │
│  Score: None                                                                                                    │
│  Published Date: 2026-08-14T00:00:00.000Z                                                                       │
│  Author: PTI                                                                                                    │
│  Image:                                                                                                         │
│  https://bl-i.thgim.com/public/incoming/r1j5e5/article70105390.ece/alternates/LANDSCAPE_1200/BL05_Tax.jpg       │
│  Favicon: None                                                                                                  │
│  Extras: None                                                                                                   │
│  Subpages: None                                                                                                 │
│  Text: Greater Chennai Corporation puts property tax revision on hold following taxpayer complaints - The       │
│  HinduBusinessLine                                                                                              │
│                                                                                                                 │
│  # Greater Chennai Corporation puts property tax revision on hold following taxpayer complaints                 │
│                                                                                                                 │
│  ## Those who have already paid the revised demand, the excess amount will be adjusted as advance tax against   │
│  subsequent half-yearly dues, the GCC said                                                                      │
│                                                                                                                 │
│  ### By PTI                                                                                                     │
│                                                                                                                 │
│   Updated - August 14, 2026 at 11:15 AM.                                                                        │
│                                                                                                                 │
│   | Chennai, Aug 14                                                                                             │
│                                                                                                                 │
│   The revised tax demand will be rolled back to the amount applicable before the revision | Photo Credit:       │
│  Shutthiphong Chandaeng                                                                                         │
│                                                       

From the output of the previous cell check all the outputs for each task. Do they match what you expected? If not, go back and refine the `expected_output`. 

You can also print the final report to see the final result of the crew

In [ ]:
from IPython.display import Markdown
Markdown(result.raw) 

You made it to the end of the lab! You can go back and experiment with the goals and backstories of the agents, as well as description and expected outputs of tasks. You can also change the inputs to any research topic you wish. Have fun with it!